<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/SUYAIBALSIFAT/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 152 (delta 61), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 1.87 MiB | 8.18 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/flyrank-ml-internship


## 1. Two paper findings + my methodology questions

Finding: "What Predicts Health?" (ML Appendix — Feature Importance). Random Forest ranks Average Position (43%) and Impressions (32%) as the top predictors of Health Score.

My methodology question: Health Score is explicitly defined earlier in the paper as impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts) — so Average Position and Impressions are literally components used to COMPUTE the label the model is predicting, not independent signals about it. This is the same label-derived-feature pattern from our own leakage lesson (Week 3/6): when a feature is a direct ingredient of the label, high importance is guaranteed, not informative. To the paper's credit, it flags this itself ("importance is descriptive rather than causal") — my question is whether the appendix could go one step further and re-run importance with the label-derived features (position, impressions, CTR, scroll) excluded, to show what (if anything) predicts health from genuinely independent signals like word count or freshness.

Finding: "The Freshness Multiplier" — the 361+ day freshness bucket shows a 283:1 growth-to-decline ratio.

My methodology question: the paper itself discloses that this ratio comes from only 1 declining page in that bucket, which is exactly the kind of small-n instability the validation design should guard against (the paper's own evidence standard sets a minimum n=50 per bucket for ML appendix analysis). My question: since the 283:1 number is still shown as a headline card at the top of the finding, alongside the honest 31-90d figure of 7.88:1, is there a risk a reader skims the card without reaching the caveat paragraph below it? A possible improvement: omit the unstable ratio from the headline card entirely, or replace it with a footnote-style "insufficient sample" marker directly on the number itself, since the paper already applies this kind of care elsewhere (e.g., the 361+ x 361+ heatmap cell, correctly flagged as survivor-biased).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

BEFORE (naive random split): precision@50 = 1.0 — a perfect score.
AFTER (grouped split by client_id): precision@50 = 0.66.
Base rate was the same in both test sets (~0.49), so this gap isn't a fluke of an easier test set — it's real memorization. With a random split, rows from the same client end up in both train and test, so the model can partly "recognize" a client's pattern rather than learning something that generalizes to a NEW client. The grouped split's 0.66 is the honest number — the random split's 1.0 was an illusion.

In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = ((df.trend_direction == "down") & (df.impressions_90d >= 100)).astype(int)

feature_cols = ["search_volume","competition","cpc","word_count","char_count",
    "impressions_90d","clicks_90d","sessions_90d","users_90d","engaged_sessions_90d",
    "ai_sessions_90d","scroll_events_90d","days_with_impressions","days_with_sessions",
    "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct"]

data = df.dropna(subset=feature_cols + ["client_id"]).copy()
X = data[feature_cols].fillna(0)
y = data["needs_review"]
groups = data["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_r = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr_r, ytr_r)
p50_random = precision_at_k(rf_r.predict_proba(Xte_r)[:, 1], yte_r.values, 50)

# AFTER: grouped split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
Xtr_g, Xte_g = X.iloc[tr_idx], X.iloc[te_idx]
ytr_g, yte_g = y.iloc[tr_idx], y.iloc[te_idx]
rf_g = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr_g, ytr_g)
p50_grouped = precision_at_k(rf_g.predict_proba(Xte_g)[:, 1], yte_g.values, 50)

print(f"BEFORE (random split) precision@50: {p50_random:.3f}  (base rate {yte_r.mean():.3f})")
print(f"AFTER  (grouped split) precision@50: {p50_grouped:.3f}  (base rate {yte_g.mean():.3f})")


BEFORE (random split) precision@50: 1.000  (base rate 0.491)
AFTER  (grouped split) precision@50: 0.660  (base rate 0.491)


## 3. Leakage audit

Ran the attack test from Week 3: trained once WITH impressions_90d (my most label-adjacent feature, since it's also part of the label's threshold condition), once WITHOUT it. Score stayed identical (0.66 both ways) — no collapse, so this is NOT a leaking feature; the model isn't secretly relying on it to reconstruct the label. I also confirmed no product flags (health_score, priority_score) or future-window columns exist anywhere in my feature list — they were never in this dataset. Timeline check: all my features (impressions_90d, ctr, avg_position, etc.) are current-state measurements, not future data.

In [3]:
cols_without = [c for c in feature_cols if c != "impressions_90d"]
rf_without = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr_g[cols_without], ytr_g)
p50_without = precision_at_k(rf_without.predict_proba(Xte_g[cols_without])[:, 1], yte_g.values, 50)

print(f"WITH impressions_90d:    precision@50 = {p50_grouped:.3f}")
print(f"WITHOUT impressions_90d: precision@50 = {p50_without:.3f}")

WITH impressions_90d:    precision@50 = 0.660
WITHOUT impressions_90d: precision@50 = 0.660


## 4. Claim rewrite

Boldest claim I could have made: "My model achieves 100% precision at identifying pages that need review." — this was the random-split number, and it's false/misleading; it reflects memorization, not real skill.

Rewritten in safe language: "Under a client-grouped validation split — the honest test of performance on clients the model has never seen — the model achieved an observed precision@50 of 0.66, meaning 66% of its top 50 flagged pages matched the review-worthy proxy label on held-out clients. This is directional and decision-support only: it ranks candidates for a human reviewer, it does not prove the flagged pages will improve if refreshed."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.